# GO enrichment results, exploratory data analysis

Generic and species-agnostic. Every section below reads only from the
fixed output schema `enrichment.output.write_results()` always produces
(`go_id, class, population, observed, expected, fold_enrichment, p_value,
q_value, significance`), plus, optionally, an `.obo` file for GO term
names and a second results file for a full-GO vs GO-slim comparison.
Point the Configuration cell at any real run's output and everything
below works, regardless of species, metric, or labeling strategy.

Per the main README's language-constraint principle (section 4): results
at the p<0.01 uncorrected threshold should be read as genes that *tended
to include* a term, not as statistically significant findings. This
notebook is for exploration and sanity-checking, not for generating
publication-ready significance claims on its own.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration -- the only cell that needs editing to point this notebook
# at a different run. Everything below is generic.
# ---------------------------------------------------------------------------

# Required: path to a summary.*.all.tsv file, e.g.
# "../results/GO/summary.CR_3D.PR_gene.all.tsv"
RESULTS_ALL_TSV = "../results/GO/summary.CR_3D.PR_gene.all.tsv"

# Optional: a second results file for the SAME labeled genes, to compare
# against the first (e.g. the GO-slim run of the same dataset). Set to
# None to skip the comparison section entirely.
RESULTS_ALL_TSV_COMPARE = "../results/GO_slim/summary.CR_3D.PR_gene.all.tsv"
COMPARE_LABEL_PRIMARY = "full-GO"
COMPARE_LABEL_SECONDARY = "GO-slim"

# Optional: path to the .obo file the run's .godb was built from, for GO
# term name/namespace lookup. Set to None to skip name lookup (tables
# will show go_id only).
OBO_PATH = "../data/go_reference/go-basic.obo"

# Which column defines "significant" for this notebook's highlighting --
# should match whatever --thresh-type the run actually used. The
# 'significance' column already reflects this; this only controls the
# volcano plot's y-axis and the near-miss calculation below.
SIG_COL = "p_value"  # or "q_value"
SIG_THRESH = 0.01

# A term with SIG_COL between SIG_THRESH and SIG_THRESH * NEAR_MISS_MULTIPLIER
# is flagged as a near miss, worth a second look rather than silently
# dropped once significance filtering happens.
NEAR_MISS_MULTIPLIER = 3

# How many top terms per class to show in the summary table.
TOP_N = 15

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_ALL_TSV, sep="\t", comment="#")
print(f"Loaded {len(df)} (class, GO term) rows from {RESULTS_ALL_TSV}")
print(f"Classes present: {sorted(df['class'].unique())}")

compare_df = None
if RESULTS_ALL_TSV_COMPARE:
    compare_df = pd.read_csv(RESULTS_ALL_TSV_COMPARE, sep="\t", comment="#")
    print(f"Loaded {len(compare_df)} rows from {RESULTS_ALL_TSV_COMPARE} for comparison")

df.head()

## GO term names

`write_results()`'s output only carries `go_id`. This section joins in
human-readable term names and namespaces from the `.obo` file, if given,
so every table below is actually readable rather than a wall of `GO:xxxxxxx`
identifiers.

In [ ]:
godag = None
if OBO_PATH:
    from goatools.obo_parser import GODag
    godag = GODag(OBO_PATH)

def add_term_names(frame):
    frame = frame.copy()
    if godag is None:
        frame["go_name"] = frame["go_id"]
        frame["go_namespace"] = ""
        return frame
    frame["go_name"] = frame["go_id"].apply(
        lambda g: godag[g].name if g in godag else "(obsolete or not in this obo)"
    )
    frame["go_namespace"] = frame["go_id"].apply(
        lambda g: godag[g].namespace if g in godag else ""
    )
    return frame

df = add_term_names(df)
if compare_df is not None:
    compare_df = add_term_names(compare_df)

df.head()

## Summary counts per class

A sanity check before anything else: does the number of significant terms
per class look plausible, or is something suspicious (e.g. every row
identically significant, or zero rows tested at all)?

In [ ]:
summary_rows = []
for cls, cls_df in df.groupby("class"):
    n_sig = int(cls_df["significance"].sum())
    n_over = int((cls_df["significance"] & (cls_df["fold_enrichment"] > 0)).sum())
    n_under = int((cls_df["significance"] & (cls_df["fold_enrichment"] < 0)).sum())
    summary_rows.append({
        "class": cls,
        "n_terms_tested": len(cls_df),
        "n_significant": n_sig,
        "n_over": n_over,
        "n_under": n_under,
        "min_p_value": cls_df["p_value"].min(),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if summary_df["n_significant"].sum() == 0:
    print(
        "No significant terms in any class. Before concluding this is a "
        "real absence of signal, check the min_p_value column above and "
        "the near-miss section below -- if the minimum p_value is close "
        "to the threshold rather than close to 1.0, a coarser vocabulary "
        "(e.g. GO-slim) may simply not be isolating a real, narrower "
        "signal visible in a finer one, not a broken run."
    )

## Top terms per class

Sorted by p-value, not filtered to significance only, so a near-miss term
is still visible here even before the dedicated near-miss section below.

In [ ]:
display_cols = ["go_id", "go_name", "go_namespace", "population", "observed",
                 "expected", "fold_enrichment", "p_value", "q_value", "significance"]

for cls, cls_df in df.groupby("class"):
    print(f"\n--- class: {cls} ---")
    display(cls_df.sort_values("p_value").head(TOP_N)[display_cols])

## Near-miss terms

Terms with `p_value` between `SIG_THRESH` and `SIG_THRESH * NEAR_MISS_MULTIPLIER`
that did NOT cross the significance cutoff. Worth a second look, especially
biologically meaningful terms sitting just outside the line -- an
uncorrected threshold is somewhat arbitrary by construction (see the main
README's reasoning on why p<0.01 uncorrected was chosen over a corrected
q-value), so a term at p=0.0103 is not meaningfully different from one at
p=0.0099, even though only one gets flagged `significance=True`.

In [ ]:
near_miss_upper = SIG_THRESH * NEAR_MISS_MULTIPLIER
near_miss = df[
    (~df["significance"])
    & (df[SIG_COL] >= SIG_THRESH)
    & (df[SIG_COL] <= near_miss_upper)
].sort_values(SIG_COL)

print(f"{len(near_miss)} near-miss terms "
      f"({SIG_THRESH} <= {SIG_COL} <= {near_miss_upper}, not already significant)")
display(near_miss[["class"] + display_cols])

## Volcano-style plot

`fold_enrichment` (x-axis, log2 scale already, per `enrichment.ora`) versus
`-log10(p_value)` (y-axis), one panel per class. Points crossing the
significance threshold are highlighted; near-miss terms (see above) get
their own marker so they are visually distinct from both the significant
and clearly non-significant points.

In [ ]:
classes = sorted(df["class"].unique())
fig, axes = plt.subplots(1, len(classes), figsize=(6 * len(classes), 5), squeeze=False)
axes = axes[0]

for ax, cls in zip(axes, classes):
    cls_df = df[df["class"] == cls].copy()
    cls_df["neg_log10_p"] = -np.log10(cls_df["p_value"].clip(lower=1e-300))

    is_near_miss = (
        (~cls_df["significance"])
        & (cls_df[SIG_COL] >= SIG_THRESH)
        & (cls_df[SIG_COL] <= near_miss_upper)
    )
    not_sig = cls_df[~cls_df["significance"] & ~is_near_miss]
    near = cls_df[is_near_miss]
    sig = cls_df[cls_df["significance"]]

    ax.scatter(not_sig["fold_enrichment"], not_sig["neg_log10_p"],
               color="0.8", s=15, label="not significant", zorder=1)
    ax.scatter(near["fold_enrichment"], near["neg_log10_p"],
               color="#E8D3A0", edgecolor="#6B5527", s=30, label="near miss", zorder=2)
    ax.scatter(sig["fold_enrichment"], sig["neg_log10_p"],
               color="#D97B3F", s=35, label="significant", zorder=3)

    ax.axhline(-np.log10(SIG_THRESH), color="0.4", linestyle="--", linewidth=0.8)
    ax.axvline(0, color="0.4", linestyle="--", linewidth=0.8)
    ax.set_xlabel("fold_enrichment (log2)")
    ax.set_ylabel("-log10(p_value)")
    ax.set_title(f"class: {cls}")
    ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
plt.show()

## Observed vs expected

A direct check on the raw counts behind each term, useful for catching
anything that looks structurally off (e.g. a population count of 0, which
should never happen since `run_ora` only tests terms with `n_pop > 0`).

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(6 * len(classes), 5), squeeze=False)
axes = axes[0]

for ax, cls in zip(axes, classes):
    cls_df = df[df["class"] == cls]
    colors = np.where(cls_df["significance"], "#D97B3F", "0.7")
    ax.scatter(cls_df["expected"], cls_df["observed"], c=colors, s=20)
    max_val = max(cls_df["expected"].max(), cls_df["observed"].max())
    ax.plot([0, max_val], [0, max_val], color="0.4", linestyle="--", linewidth=0.8,
            label="observed = expected")
    ax.set_xlabel("expected")
    ax.set_ylabel("observed")
    ax.set_title(f"class: {cls}")
    ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
plt.show()

## p-value distribution

A rough overall sanity check independent of any single term: under a null
with no real enrichment signal at all, p-values would be roughly uniform
across [0, 1]. A pileup near 0 suggests real signal; a pileup near 1 across
every class, with none near 0, would be a sign to double-check the run
rather than assume genuine absence of enrichment.

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(6 * len(classes), 4), squeeze=False)
axes = axes[0]

for ax, cls in zip(axes, classes):
    cls_df = df[df["class"] == cls]
    ax.hist(cls_df["p_value"], bins=20, range=(0, 1), color="#B7BE8D", edgecolor="#4A4D2E")
    ax.axvline(SIG_THRESH, color="#D97B3F", linestyle="--", linewidth=1)
    ax.set_xlabel("p_value")
    ax.set_ylabel("number of GO terms")
    ax.set_title(f"class: {cls}")

fig.tight_layout()
plt.show()

## Comparison against a second run

Runs only if `RESULTS_ALL_TSV_COMPARE` is set. Intended for comparing the
same labeled genes against two different `.godb` files (e.g. full-GO vs
GO-slim), but works for any two result files that share `go_id`/`class`
values, e.g. two different labeling strategies on the same population.

In [ ]:
if compare_df is not None:
    merged = df.merge(
        compare_df, on=["go_id", "class"], suffixes=("_primary", "_secondary"), how="inner"
    )
    print(f"{len(merged)} (class, go_id) pairs present in both "
          f"{COMPARE_LABEL_PRIMARY} ({len(df)} rows) and "
          f"{COMPARE_LABEL_SECONDARY} ({len(compare_df)} rows)")

    fig, ax = plt.subplots(figsize=(6, 6))
    colors = np.where(
        merged["significance_primary"] | merged["significance_secondary"],
        "#D97B3F", "0.7"
    )
    ax.scatter(merged["fold_enrichment_primary"], merged["fold_enrichment_secondary"],
               c=colors, s=20)
    lims = [
        min(merged["fold_enrichment_primary"].min(), merged["fold_enrichment_secondary"].min()),
        max(merged["fold_enrichment_primary"].max(), merged["fold_enrichment_secondary"].max()),
    ]
    ax.plot(lims, lims, color="0.4", linestyle="--", linewidth=0.8)
    ax.set_xlabel(f"fold_enrichment ({COMPARE_LABEL_PRIMARY})")
    ax.set_ylabel(f"fold_enrichment ({COMPARE_LABEL_SECONDARY})")
    ax.set_title("Shared terms: fold_enrichment agreement between the two runs")
    fig.tight_layout()
    plt.show()

    only_primary_sig = merged[merged["significance_primary"] & ~merged["significance_secondary"]]
    only_secondary_sig = merged[merged["significance_secondary"] & ~merged["significance_primary"]]
    print(f"\nSignificant in {COMPARE_LABEL_PRIMARY} only: {len(only_primary_sig)}")
    if len(only_primary_sig):
        display(only_primary_sig[["class", "go_id", "go_name_primary",
                                    "p_value_primary", "p_value_secondary"]])
    print(f"\nSignificant in {COMPARE_LABEL_SECONDARY} only: {len(only_secondary_sig)}")
    if len(only_secondary_sig):
        display(only_secondary_sig[["class", "go_id", "go_name_primary",
                                      "p_value_primary", "p_value_secondary"]])
else:
    print("RESULTS_ALL_TSV_COMPARE is not set, skipping comparison section.")

## Auto-generated plain-language summary

Templated, not hand-written per dataset -- reruns correctly for any input.
Uses the main README's mandated language ("tended to include", not
"statistically significant") since this threshold applies no
multiple-testing correction. Treat this as a first draft to edit, not a
final write-up.

In [ ]:
for cls, cls_df in df.groupby("class"):
    sig_over = cls_df[cls_df["significance"] & (cls_df["fold_enrichment"] > 0)].sort_values("p_value")
    sig_under = cls_df[cls_df["significance"] & (cls_df["fold_enrichment"] < 0)].sort_values("p_value")

    print(f"Class '{cls}' ({len(cls_df)} terms tested):")
    if len(sig_over):
        names = ", ".join(sig_over["go_name"].head(5))
        print(f"  Genes in this class tended to include: {names}"
              f"{' (and others)' if len(sig_over) > 5 else ''}.")
    if len(sig_under):
        names = ", ".join(sig_under["go_name"].head(5))
        print(f"  Genes in this class tended to under-represent: {names}"
              f"{' (and others)' if len(sig_under) > 5 else ''}.")
    if not len(sig_over) and not len(sig_under):
        print(f"  No terms reached the significance threshold for this class.")
    print()

## Next steps

This notebook is exploration only, it never writes anything to disk. For
any GO ID, the full definition (not just name/namespace) can be looked up
at https://amigo.geneontology.org/amigo/term/<go_id>.